In [ ]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import tensorflow as tf
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


# ==============================
# 2. DROWSINESS MODEL (CNN)
# ==============================
print("\n🚀 Training Drowsiness Model...")

drowsy_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_drowsy = drowsy_datagen.flow_from_directory(
    "../datasets/Drowsiness",
    target_size=(128,128),
    batch_size=16,
    class_mode='binary',
    subset='training'
)

val_drowsy = drowsy_datagen.flow_from_directory(
    "../datasets/Drowsiness",
    target_size=(128,128),
    batch_size=16,
    class_mode='binary',
    subset='validation'
)

drowsy_model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128,activation='relu'),
    Dense(1,activation='sigmoid')
])

drowsy_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
drowsy_model.fit(train_drowsy, epochs=3, validation_data=val_drowsy)


# ==============================
# 3. POTHOLE MODEL (CNN)
# ==============================
print("\n🚀 Training Pothole Model...")

pothole_datagen = ImageDataGenerator(rescale=1./255)

train_pothole = pothole_datagen.flow_from_directory(
    "../datasets/potholes/train",
    target_size=(128,128),
    batch_size=16,
    class_mode='categorical'
)

pothole_model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128,activation='relu'),
    Dense(2,activation='softmax')
])

pothole_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
pothole_model.fit(train_pothole, epochs=3)


# ==============================
# 4. LIGHTING MODEL (CSV - ML)
# ==============================
print("\n🚀 Training Lighting Model...")

data = pd.read_csv("../datasets/light/street_light_fault_prediction_dataset.csv")

# Clean data
data = data.fillna(0)

print("Columns:", data.columns)

# 👉 CHANGE 'fault' if needed
target_column = data.columns[-1]   # automatically take last column

X = data.drop(target_column, axis=1)
y = data[target_column]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

light_model = RandomForestClassifier()
light_model.fit(X_train, y_train)

print("Lighting Model Ready ✅")


# ==============================
# 5. PREDICTION FUNCTIONS
# ==============================
def predict_drowsiness(img_path):
    img = load_img(img_path, target_size=(128,128))
    img_array = img_to_array(img) / 255
    img_array = np.expand_dims(img_array, axis=0)

    result = drowsy_model.predict(img_array)
    return "drowsy" if result[0][0] > 0.5 else "normal"


def predict_pothole(img_path):
    img = load_img(img_path, target_size=(128,128))
    img_array = img_to_array(img) / 255
    img_array = np.expand_dims(img_array, axis=0)

    result = pothole_model.predict(img_array)
    return "pothole" if np.argmax(result) == 0 else "normal"


def predict_lighting(input_data):
    input_array = np.array(input_data).reshape(1, -1)
    result = light_model.predict(input_array)

    return "poor_light" if result[0] == 1 else "good_light"


# ==============================
# 6. FINAL DECISION ENGINE
# ==============================
def final_decision(img_path, lighting_input):

    print("\n🔍 ANALYZING...")

    drowsy = predict_drowsiness(img_path)
    pothole = predict_pothole(img_path)
    lighting = predict_lighting(lighting_input)

    print("\n📊 RESULTS:")
    print("Drowsiness:", drowsy)
    print("Pothole:", pothole)
    print("Lighting:", lighting)

    # 🚨 FINAL LOGIC
    if pothole == "pothole" or lighting == "poor_light":
        print("\n⚠ ROAD ALERT!")
        print("➡ Show danger symbol on map")
        print("➡ Get municipal corporation contact")

    elif drowsy == "drowsy":
        print("\n🔊 DRIVER ALERT!")
        print("➡ Play warning sound")
        print("➡ Call ambulance (108)")

    else:
        print("\n✅ SAFE CONDITION")


# ==============================
# 7. TEST YOUR SYSTEM
# ==============================

# 👉 Replace with real values from your dataset columns
sample_lighting_input = X.iloc[0].tolist()

final_decision("../test.jpg", sample_lighting_input)